---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

---

## Group G35 — Pistar

- Ariel Cornelius Sitorus (13524085)
- Rava Khoman Tuah Saragih (13524107)
- Natanael I Manurung (13524021)
- Rainaldi Pratama Sembiring (13524117)
- Raynard Fausta (13524052)


---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

Proyek ini menggunakan data booking hotel untuk memprediksi pembatalan.

## Gambaran persoalan

Prediksi pembatalan booking membantu penginapan memperkirakan okupansi dan menentukan booking yang perlu dikonfirmasi. Setiap baris memuat informasi tamu, lama menginap, paket makan, kategori kamar, kanal pemesanan, harga, dan riwayat booking. Target 1 berarti batal, target 0 berarti tetap berlangsung.


## Aim

Tujuan proyek ini adalah memprediksi `is_canceled` untuk setiap booking. Nilai `1` berarti batal dan `0` berarti tidak batal. Kami membuat lima model klasifikasi from scratch dan membandingkannya dengan model scikit-learn.

## Metrik

Metrik resmi kompetisi belum dapat diverifikasi: tautan undangan Kaggle pada spesifikasi tidak dapat diakses saat audit. Macro F1 dipakai sementara untuk pemilihan model, yaitu rata-rata F1 kelas batal dan tidak batal. Accuracy, F1 positif, precision, recall, ROC-AUC, PR-AUC, confusion matrix, serta waktu training/prediksi juga dilaporkan. Angka validasi lokal bukan skor leaderboard.


## Dataset

`train.csv` berisi 27.373 booking dengan label, sedangkan `test.csv` berisi 6.844 booking yang perlu diprediksi. Fiturnya mencakup informasi tamu, lama menginap, tipe kamar, paket makan, kanal booking, harga dalam rupiah, dan riwayat booking. Beberapa nilai fitur kosong. `sample_submission.csv` menunjukkan format `id,is_canceled`.

## Pendekatan

Lima model scratch dibandingkan menggunakan fold development yang sama. Preprocessing dipelajari ulang dalam training setiap fold. Parameter dipilih dari CV development, threshold dari OOF development, lalu holdout dipakai untuk evaluasi akhir. Kandidat final direfit pada seluruh train.csv untuk menghasilkan submission. Rincian batas training subset dan perbedaan representasi tercantum pada bagian tuning dan perbandingan.


---

# Initialization <a name="2"></a>

---

Bagian ini menyiapkan environment, konfigurasi, dan berkas data.


## Environment Setup

Instal dependensi dari `requirements.txt`, lalu simpan ketiga file CSV kompetisi di folder `data/`.

## Import Libraries

Impor library yang diperlukan untuk menyiapkan notebook dan membaca data.

In [ ]:
import random
from pathlib import Path
from dataclasses import asdict, replace
import json
import hashlib
import sys
import time
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as NotebookImage
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from threadpoolctl import threadpool_limits
get_ipython().run_line_magic('matplotlib', 'inline')


## Settings

Cari folder utama proyek supaya data bisa dibaca saat notebook dijalankan dari folder utama atau dari folder `notebooks/`.

In [ ]:
current_dir = Path.cwd().resolve()
for directory in (current_dir, *current_dir.parents):
    if (directory / 'src').is_dir() and (directory / 'requirements.txt').is_file():
        project_root = directory
        break
else:
    raise FileNotFoundError('Jalankan notebook dari dalam folder proyek.')
sys.path.insert(0, str(project_root))
class Settings:
    SEED = 42
    DATA_DIR = project_root / 'data'
    OUTPUT_DIR = project_root / 'output'
    ID_COLUMN = 'id'
Settings.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for folder in ('laporan/gambar', 'laporan/tabel', 'laporan/animasi'):
    (Settings.OUTPUT_DIR / folder).mkdir(parents=True, exist_ok=True)
from src.models import LogisticRegressionScratch, SVMScratch
from src.preprocessing import (NUMERIC, CATEGORICAL, FEATURES, SKEWED, TARGET,
                               BookingPreprocessor, clean_features, add_features, feature_groups)
from src.experiments import (Search, baseline_candidates, split_development_holdout,
                             run_search, evaluate_holdout, fit_candidate)
from src.evaluation import evaluate
from src.submission import save_final, predict_bundle, validate_submission


## Seed Everything

Tetapkan seed Python dan NumPy agar operasi acak dapat diulang.

In [ ]:
random.seed(Settings.SEED)
np.random.seed(Settings.SEED)
thread_limit = threadpool_limits(limits=1)


## Load Dataset

Baca ketiga file CSV kompetisi. Kolom `id` dibaca sebagai teks karena beberapa ID menyerupai angka.

In [ ]:
train = pd.read_csv(Settings.DATA_DIR / 'train.csv', dtype={'id': str})
test = pd.read_csv(Settings.DATA_DIR / 'test.csv', dtype={'id': str})
sample_submission = pd.read_csv(Settings.DATA_DIR / 'sample_submission.csv', dtype={'id': str})
input_sha256 = {name: hashlib.sha256((Settings.DATA_DIR / name).read_bytes()).hexdigest()
                for name in ('train.csv', 'test.csv', 'sample_submission.csv')}
display(pd.Series({'train': len(train), 'test': len(test), 'sample_submission': len(sample_submission)}))


---

# Data Understanding (Optional) <a name="3"></a>

---

Pemahaman data menjadi dasar keputusan cleaning dan preprocessing berikut.


## Deskripsi data

Periksa ukuran data dan tipe setiap atribut.


In [ ]:
display(train.dtypes.rename('tipe').to_frame())
print('Train:', train.shape, '| Kaggle:', test.shape)


## Eksplorasi data

Periksa distribusi kelas dan rentang nilai numerik.


In [ ]:
display(train[TARGET].value_counts().sort_index().rename('jumlah').to_frame())
display(train.select_dtypes(include='number').describe().T)


## Kualitas data

Periksa missing, konsistensi ID, serta validitas label.


In [ ]:
quality = pd.DataFrame({'missing_train': train.isna().sum(), 'missing_test': test.isna().sum()})
display(quality)
assert train.id.notna().all() and train.id.is_unique
assert test.id.notna().all() and test.id.is_unique
assert train.is_canceled.isin([0, 1]).all()


---

# Training and Validation Split <a name="4"></a>

---

Split memakai StratifiedGroupKFold, seed 42: sekitar 80% development dan 20% holdout.
Booking dengan predictor identik masuk bagian yang sama, termasuk bila labelnya berbeda.
Split tidak diubah untuk mengejar skor. Holdout yang sama sudah pernah dievaluasi pada
audit sebelumnya, sehingga hasilnya bukan estimasi dari holdout baru yang belum pernah
diamati. Fungsi tuning hanya menerima development dan tidak membaca label holdout.


In [ ]:
train_set, val_set = split_development_holdout(train, seed=Settings.SEED)
assert not set(feature_groups(train_set)).intersection(feature_groups(val_set))
print('Development:', len(train_set), '| holdout tersegel:', len(val_set))


---

# Data Preparation <a name="5"></a>

---

Setiap langkah persiapan diterapkan sesuai karakteristik data; keputusan dicatat di bawah langkah terkait.


Prapemrosesan boleh memakai scikit-learn. Seluruh statistik hanya dipelajari dari training fold, lalu diterapkan pada bagian lain.


## Pemilihan data

ID dan target dikeluarkan dari predictor.


In [ ]:
num_cols, cat_cols, feature_cols = NUMERIC, CATEGORICAL, FEATURES
display(train_set[feature_cols].dtypes.value_counts())


## Pembersihan data

Missing, duplikat, outlier, dan nilai tidak valid diperiksa terpisah.


### Nilai hilang

Imputer matrix mempelajari median dari training. Tree dan Bayes mempertahankan missing.


In [ ]:
display(train_set[feature_cols].isna().sum().rename('missing_development').to_frame())


### Duplikat

Booking berbeda ID tetap disimpan; pola predictor identik dikelompokkan saat validasi.


In [ ]:
duplicates = train_set.duplicated(subset=feature_cols).sum()
conflicts = train_set.groupby(feature_cols, dropna=False)[TARGET].nunique().gt(1).sum()
print('Pola predictor berulang:', duplicates, '| pola dengan label berbeda:', conflicts)
print('Seluruh booking dengan ID berbeda dipertahankan; pola yang sama dikelompokkan saat split.')


### Outlier

Nilai ekstrem yang masih masuk akal tidak dibuang otomatis.


In [ ]:
skewed_cols = SKEWED
q1, q3 = train_set[skewed_cols].quantile(.25), train_set[skewed_cols].quantile(.75)
iqr = q3 - q1
display(((train_set[skewed_cols] < q1 - 1.5 * iqr) |
         (train_set[skewed_cols] > q3 + 1.5 * iqr)).sum().rename('outlier_IQR'))
print('Outlier tidak dibuang otomatis. Log1p digunakan untuk view matrix.')


### Kesalahan data

Nilai numerik tidak valid, kategori kosong, dan tanggal mustahil menjadi NaN.


In [ ]:
preview = clean_features(train_set)
print('Tambahan missing setelah validasi:',
      int(preview.isna().sum().sum() - train_set[FEATURES].isna().sum().sum()))
print('Cleaning deterministik tidak membaca target atau statistik holdout/test.')


## Konstruksi fitur

Fitur turunan hanya menggunakan predictor tanpa target.


### Atribut turunan

Kelompok fitur diuji terpisah melalui ablation CV.


In [ ]:
engineered_preview = add_features(train_set, ('basic', 'stay', 'history', 'season'))
display(engineered_preview[[c for c in engineered_preview if c not in FEATURES]].describe().T)
print('Fitur basic/stay/history/season akan diuji melalui ablation CV, tidak otomatis dipakai.')


### Skala fitur

Transformasi numeric dibedakan untuk matrix, tree, dan Bayes.


In [ ]:
print('Matrix: median training, log1p pada fitur miring, standardisasi, one-hot, penanda missing.')
print('Tree: angka dan NaN dipertahankan; kategori menjadi kode diskret, unknown menjadi NaN.')
print('Bayes: numeric distandardisasi agar epsilon variance tidak didominasi nominal harga.')


### Encoding kategori

One-hot dipakai pada matrix; kode diskret serta NaN pada tree/Bayes.


In [ ]:
display(pd.DataFrame({'model': ['KNN/LR/SVM', 'Decision Tree', 'Naive Bayes'],
                      'view': ['matrix', 'tree', 'bayes'],
                      'missing': ['imputer + indicator', 'fractional branches', 'ignore missing likelihood']}))


### Reduksi dimensi dan pemilihan fitur

PCA dan MI menjadi diagnostik development; MI top25 diuji dalam training fold.


In [ ]:
probe_prep = BookingPreprocessor().fit(train_set, train_set[TARGET])
probe = probe_prep.transform(train_set)
pca = PCA().fit(probe)
needed = int(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), .95) + 1)
mi = mutual_info_classif(probe, train_set[TARGET], random_state=Settings.SEED)
print('Dimensi matrix:', probe.shape[1], '| PCA 95%:', needed,
      '| fitur MI < 0.001:', int((mi < .001).sum()))
print('PCA hanya diagnostik; MI top25 diuji sebagai kandidat LR, di-fit ulang dalam tiap fold.')


## Ketidakseimbangan kelas

Bobot balanced dihitung dari training fold dan dibandingkan dengan tanpa pembobotan.


In [ ]:
display(train_set[TARGET].value_counts(normalize=True).round(3))
sample_weight = compute_sample_weight('balanced', train_set[TARGET])
print('Bobot kelas dibandingkan dengan tanpa bobot melalui CV, bukan otomatis diterapkan.')


## Format data

Label biner dan representasi fitur disiapkan secara konsisten.


In [ ]:
y_train = train_set[TARGET].to_numpy(dtype=int)
print('Label development:', y_train.shape)
print('train.csv dan test.csv tidak pernah digabungkan untuk fitting preprocessing.')


---

# Preparation Pipeline <a name="6"></a>

---

Pipeline yang sama diterapkan saat training, validasi, serialization, dan prediksi Kaggle test.


In [ ]:
prep_matrix = BookingPreprocessor('matrix').fit(train_set, y_train)
prep_tree = BookingPreprocessor('tree').fit(train_set, y_train)
X_train_matrix = prep_matrix.transform(train_set)
X_train_tree = prep_tree.transform(train_set)
tree_cat_idx = prep_tree.categorical_idx_
num_all = prep_tree.numeric_
print('Development matrix:', X_train_matrix.shape, '| tree:', X_train_tree.shape)


---

# Modeling and Validation <a name="7"></a>

---

Target adalah is_canceled. Kelas model diimpor dari src; komputasi inti model hanya menggunakan NumPy.


## K-Nearest Neighbors

Baseline divalidasi pada fold development; k, bobot, dan metrik jarak diuji saat tuning.


In [ ]:
experiment_dir = Settings.OUTPUT_DIR / 'laporan/evaluasi'
search = Search(train_set, experiment_dir, folds=3)
baselines = {c.family: c for c in baseline_candidates()}
search.run([baselines['KNN']], 'baseline')
print('KNN: baseline k15 uniform; tuning membandingkan k, bobot jarak, Euclidean, dan Manhattan. Minkowski juga tersedia dan diuji terhadap referensi.')


## Decision Tree Learning

C4.5 (gain ratio) dan entropy menangani kategori serta missing melalui cabang fraksional.
Gini memakai split biner CART, termasuk partisi kategori berdasarkan frekuensi
kelas training. Kandidat preprocessing membandingkan missing asli, median + indikator missing, dan one-hot.


In [ ]:
search.run([baselines['Tree']], 'baseline')
print('Baseline C4.5 depth10 leaf5; entropy dan gain_ratio dituning bertahap.')


## Logistic Regression

Mendukung tanpa penalti, L1, L2, Elastic Net, GD, SGD, dan Adam.


In [ ]:
search.run([baselines['Logistic']], 'baseline')
print('LR memakai mean BCE + regularisasi. L1/Elastic Net memakai langkah proximal.')
print('GD, SGD minibatch, Adam, kekuatan regularisasi, dan batch size diuji dalam CV.')


## Support Vector Machine

Kernel Linear, Polynomial, dan Gaussian/RBF tetap tersedia.


In [ ]:
search.run([baselines['SVM']], 'baseline')
print('SVM dibatasi 2.000 baris training stratified per fold; bukan full-data SVM.')
print('Kernel penuh diperiksa batas memorinya sebelum alokasi. Kandidat memakai kalibrasi internal terpisah.')


## Naive Bayes

Likelihood Gaussian numerik dipadukan dengan probabilitas kategorikal dan smoothing.


In [ ]:
search.run([baselines['Bayes']], 'baseline')
print('Bayes campuran mempertahankan likelihood kategorikal; view scaled diuji terhadap baseline raw.')


## Evaluasi model

Nama macro_f1 dan f1_positive dibedakan. PR-AUC memakai integrasi trapezoidal, bukan average precision.


In [ ]:
baseline_cv = search.summary()
display(baseline_cv[['candidate', 'macro_f1_mean', 'macro_f1_std', 'accuracy_mean', 'roc_auc_mean']])
print('Semua angka di atas adalah CV development; holdout belum dibuka.')


## Hyperparameter tuning dan pemilihan model

Screening memakai 3-fold stratified grouped CV; sekitar delapan kandidat tree terbaik,
baseline, dan kandidat terbaik setiap keluarga dikonfirmasi pada 5-fold dengan seed 42.
Tree menguji gain ratio, entropy dan Gini, depth 8?24, leaf 1?30, split 2?40,
bobot balanced, minimum impurity decrease, pruning, serta smoothing probabilitas leaf.
Pencarian bertahap mempersempit ruang kandidat, tanpa exhaustive grid seluruh kombinasi.
Ablation mengisolasi encoding, durasi/tamu, riwayat, harga, tanggal/musiman, interaksi,
dan kombinasi kelompok yang meningkatkan CV. Semua statistik dipelajari dalam fold.
Pilihan berasal dari mean Macro F1 CV pada threshold 0,5; perubahan <= 0,0005
mempertahankan baseline. Threshold OOF memakai 0,20?0,80 langkah 0,05, dilanjutkan
refine 0,01 dan pemeriksaan kestabilan antar-fold. Metrik resmi Kaggle belum terverifikasi.
KNN/LR membandingkan subset dengan seluruh training fold; SVM memakai subset
2.000?3.000. Ukuran aktual dan train/validation gap dicatat dalam CSV.
CV/OOF merupakan hasil pencarian dan dapat optimistis. Holdout lama tidak digunakan
untuk memilih ulang konfigurasi setelah hasil evaluasi akhir diketahui.


In [ ]:
search, selected, final_candidate, final_threshold = run_search(train_set, experiment_dir, search=search)
cv_summary = search.summary()
cv_folds = pd.DataFrame(search.rows)
display(cv_summary[['candidate', 'macro_f1_mean', 'macro_f1_std', 'accuracy_mean', 'roc_auc_mean']].head(15))
print('Pilihan dikunci sebelum holdout:', final_candidate.name, '| threshold OOF:', final_threshold)
print('Metrik Kaggle belum terverifikasi: Macro F1 menjadi metrik sementara, bukan klaim metrik resmi.')
print('OOF threshold adalah hasil tuning; estimasi generalisasi berasal dari holdout berikut.')

from src.experiments import choose_threshold
stability = []
for fold, (_, validation) in enumerate(search.folds):
    fold_y = search.y[validation]
    fold_p = search.oof[final_candidate.key][validation]
    fold_threshold, _ = choose_threshold(fold_y, fold_p)
    stability.append(dict(fold=fold, optimal_diagnostic_threshold=fold_threshold,
                          applied_global_oof_threshold=final_threshold,
                          **evaluate(fold_y, fold_p, final_threshold)))
threshold_stability = pd.DataFrame(stability)
threshold_stability.to_csv(experiment_dir/'threshold_stability.csv', index=False)
display(threshold_stability[['fold','optimal_diagnostic_threshold','macro_f1']])
print('Threshold per fold di atas hanya diagnostik; threshold final tetap OOF gabungan.')


---

# Comparison with scikit-learn <a name="8"></a>

---

Kelima model scratch dibandingkan dengan scikit-learn sesudah konfigurasi final dikunci. Semua estimator pustaka hanya pembanding.


In [ ]:
holdout_scores, fitted = evaluate_holdout(train_set, val_set, search, selected)
y_val = val_set[TARGET].to_numpy(dtype=int)
display(holdout_scores[['family', 'phase', 'threshold_kind', 'threshold', 'accuracy',
                      'macro_f1', 'f1_positive', 'roc_auc', 'pr_auc', 'n_train']].round(4))

names = {'KNN': 'KNN', 'Tree': 'DTL', 'Logistic': 'Logistik', 'SVM': 'SVM', 'Bayes': 'Bayes'}
rows, predictions, probabilities, models = [], {}, {}, []
for family, candidate in selected.items():
    prep, own, p = fitted['optimized', family]
    raw = train_set
    if candidate.max_train and len(raw) > candidate.max_train:
        ids, _ = train_test_split(np.arange(len(raw)), train_size=candidate.max_train,
                                 stratify=raw[TARGET], random_state=Settings.SEED)
        raw = raw.iloc[ids]
    yt_ref = raw[TARGET].to_numpy(dtype=int)
    ref_prep = BookingPreprocessor('matrix', candidate.groups, candidate.select_k).fit(raw, yt_ref)
    A, V = ref_prep.transform(raw), ref_prep.transform(val_set)
    params = candidate.params
    if family == 'Tree':
        ref = DecisionTreeClassifier(criterion='gini' if params.get('criterion') == 'gini' else 'entropy', max_depth=params.get('max_depth'),
                                    min_samples_leaf=params.get('min_samples_leaf', 1),
                                    min_samples_split=params.get('min_samples_split', 2),
                                    min_impurity_decrease=params.get('min_impurity_decrease', 0.),
                                    ccp_alpha=max(.0001, candidate.ccp_alpha) if params.get('criterion', 'gain_ratio') == 'gain_ratio' else candidate.ccp_alpha,
                                    random_state=Settings.SEED)
    elif family == 'KNN':
        ref = KNeighborsClassifier(n_neighbors=params.get('k', 15),
                    weights='distance' if params.get('weighted', False) else 'uniform',
                    metric=params.get('metric', 'euclidean'), p=params.get('p', 3))
    elif family == 'Logistic':
        penalty, strength = params.get('penalty'), params.get('reg_strength', 0)
        extra = {'l1_ratio': params.get('l1_ratio', .5)} if penalty == 'elasticnet' else {}
        ref = LogisticRegression(penalty=penalty, solver='saga' if penalty in ('l1', 'elasticnet') else 'lbfgs',
                                 C=1/(len(raw)*strength) if strength else 1.,
                                 max_iter=3000, tol=1e-5, random_state=Settings.SEED, **extra)
    elif family == 'SVM':
        ref = SVC(kernel=params.get('kernel', 'rbf'), C=params.get('C', 1),
                  degree=params.get('degree', 3), coef0=params.get('coef0', 1),
                  gamma=params.get('gamma', 'scale'), probability=True, random_state=Settings.SEED)
    else:
        ref = GaussianNB(var_smoothing=params.get('var_smoothing', 1e-9))
    name = names[family]
    own_row = holdout_scores.query("family == @family and phase == 'optimized' and threshold_kind == 'default'").iloc[0].to_dict()
    rows.append(dict(own_row, model=name, versi='Mandiri', macro=own_row['macro_f1']))
    started = time.perf_counter()
    weights = compute_sample_weight('balanced', yt_ref) if candidate.balanced else None
    if family == 'KNN': ref.fit(A, yt_ref)
    else: ref.fit(A, yt_ref, sample_weight=weights)
    trained = time.perf_counter()
    lib_p = ref.predict_proba(V)[:, 1]
    rows.append(dict(model=name, versi='Pustaka', n_train=len(raw), fit_time=trained-started,
                     predict_time=time.perf_counter()-trained, macro=evaluate(y_val, lib_p)['macro_f1'],
                     **evaluate(y_val, lib_p)))
    for impl, proba in [('Mandiri', p), ('Pustaka', lib_p)]:
        probabilities[name, impl], predictions[name, impl] = proba, (proba >= .5).astype(int)
    models.append((name, own, ref, (None, prep.transform(val_set)), (A, V)))
scores = pd.DataFrame(rows)
scores.to_csv(Settings.OUTPUT_DIR / 'laporan/tabel/results.csv', index=False)
display(scores[['model','versi','accuracy','macro_f1','f1_positive','roc_auc','pr_auc','fit_time']].round(4))
print('Tree/Bayes menggunakan representasi berbeda dari estimator pustaka; lihat view masing-masing.')
print('SVM scratch dan SVC memakai prosedur kalibrasi berbeda; jenis scratch tercatat pada model.')

idx, _ = train_test_split(np.arange(len(train_set)), train_size=min(2000, len(train_set)-1),
                          stratify=y_train, random_state=Settings.SEED)
xt, yt = train_set.iloc[idx], y_train[idx]
small_matrix = BookingPreprocessor('matrix').fit(xt, yt)
M = (small_matrix.transform(xt), small_matrix.transform(val_set))
variant_rows = []
for penalty in (None, 'l1', 'l2', 'elasticnet'):
    own = LogisticRegressionScratch(optimizer='adam', learning_rate=.03, penalty=penalty,
                                    reg_strength=.001 if penalty else 0, max_iter=500)
    ref = LogisticRegression(penalty=penalty, solver='saga' if penalty in ('l1','elasticnet') else 'lbfgs',
                             C=1/(len(yt)*.001) if penalty else 1., max_iter=3000, tol=1e-5,
                             random_state=Settings.SEED, **({'l1_ratio': .5} if penalty == 'elasticnet' else {}))
    for impl, model in [('Mandiri', own), ('Pustaka', ref)]:
        model.fit(M[0], yt)
        variant_rows.append(dict(model='Logistik '+str(penalty), versi=impl,
                                 **evaluate(y_val, model.predict_proba(M[1])[:,1])))
for kernel in ('linear','poly','rbf'):
    for impl, model in [('Mandiri', SVMScratch(kernel=kernel, calibration_fraction=.15)),
                        ('Pustaka', SVC(kernel=kernel, coef0=1, probability=True, random_state=Settings.SEED))]:
        model.fit(M[0], yt)
        variant_rows.append(dict(model='SVM '+kernel, versi=impl,
                                 **evaluate(y_val, model.predict_proba(M[1])[:,1])))
variant_scores = pd.DataFrame(variant_rows)
variant_scores.to_csv(Settings.OUTPUT_DIR/'laporan/tabel/variants.csv', index=False)
display(variant_scores.round(4))


## Decision Tree pada seluruh development

Pohon terpilih dari CV dilatih memakai semua booking development. Metrik berikut
berasal dari holdout berlabel; angka ini tidak digunakan lagi untuk tuning.


In [ ]:
prep_tree, tree_full, tree_p = fitted['optimized', 'Tree']
X_train_tree, X_val_tree = prep_tree.transform(train_set), prep_tree.transform(val_set)
num_all, tree_cat_idx = prep_tree.numeric_, prep_tree.categorical_idx_
full_pred = {'Mandiri': predictions['DTL','Mandiri'], 'Pustaka': predictions['DTL','Pustaka']}
full_scores = scores.query("model == 'DTL'").copy()
full_scores.to_csv(Settings.OUTPUT_DIR/'laporan/tabel/full_tree_results.csv', index=False)
display(full_scores[['versi','n_train','accuracy','macro_f1','roc_auc']].round(4))

print('Train/holdout Macro F1 pada threshold 0.5:',
      evaluate(y_train, tree_full.predict_proba(X_train_tree)[:,1])['macro_f1'],
      evaluate(y_val, tree_p)['macro_f1'])


## Gambar 1-4

Gambar 2 dan 3 memakai holdout sesudah pemilihan parameter dikunci. Jumlah training
masing-masing model mengikuti anggaran yang ditulis di tabel; pohon memakai seluruh development.


In [ ]:
from src.bonus.tree_viz import plot_tree
out = Settings.OUTPUT_DIR
ax = train[TARGET].value_counts().sort_index().plot.bar(rot=0)
ax.set(xticklabels=['Tidak batal', 'Batal'], ylabel='Jumlah reservasi')
ax.bar_label(ax.containers[0])
ax.figure.savefig(out / 'laporan/gambar/gambar_1_kelas.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(ax.figure)
ax = scores.pivot(index='model', columns='versi', values='macro').plot.bar(rot=0)
ax.set(xlabel='Algoritma', ylabel='F1 makro', ylim=(0, 1))
ax.figure.savefig(out / 'laporan/gambar/gambar_2_f1.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(ax.figure)
fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout='constrained')
matrices = {version: confusion_matrix(y_val, pred, labels=[0, 1]) for version, pred in full_pred.items()}
maximum = max(int(matrix.max()) for matrix in matrices.values())
for ax, (version, matrix) in zip(axes, matrices.items()):
    ConfusionMatrixDisplay(matrix, display_labels=['Tidak batal', 'Batal']).plot(
        ax=ax, cmap='Blues', colorbar=False, values_format='d', im_kw={'vmin': 0, 'vmax': maximum})
    ax.set(title=version, xlabel='Prediksi', ylabel='Aktual')
fig.savefig(out / 'laporan/gambar/gambar_3_kesalahan.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
levels = {col: list(values) for col, values in zip(cat_cols, prep_tree.columns_.named_transformers_['cat'].categories_)}
fig = plot_tree(tree_full, top_n=2, feature_names=prep_tree.feature_names_,
                category_levels=levels, path=out / 'laporan/gambar/gambar_4_pohon.png')
plt.show()
plt.close(fig)


## Blok E1 - Pemangkasan pohon

Model bonus di-fit pada training internal development. Calibration memilih alpha dan
check menilai hasilnya; ketiganya memisahkan grup predictor identik. Holdout final
bukan sumber pemilihan alpha atau threshold bonus.


In [ ]:
from src.bonus.tree_prune import tune_ccp_alpha, plot_pruning
from sklearn.model_selection import StratifiedGroupKFold
bonus_training, bonus_validation = search.folds[0]
bonus_raw = train_set.iloc[bonus_training]
bonus_validation_raw = train_set.iloc[bonus_validation].reset_index(drop=True)
bonus_prep, bonus_tree, _ = fit_candidate(selected['Tree'], bonus_raw)
bonus_X = bonus_prep.transform(bonus_validation_raw)
bonus_y = bonus_validation_raw[TARGET].to_numpy(dtype=int)
cal, check = next(StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=Settings.SEED).split(
    bonus_validation_raw, bonus_y, feature_groups(bonus_validation_raw)))
assert not set(feature_groups(bonus_raw)).intersection(feature_groups(bonus_validation_raw))
assert not set(feature_groups(bonus_validation_raw.iloc[cal])).intersection(feature_groups(bonus_validation_raw.iloc[check]))
pruned, pruning_path = tune_ccp_alpha(bonus_tree, bonus_X[cal], bonus_y[cal])
fig = plot_pruning(pruning_path, path=out / 'laporan/gambar/gambar_5_pemangkasan.png')
plt.show()
plt.close(fig)
best = pruning_path[pruning_path.roc_auc.round(12) == pruning_path.roc_auc.round(12).max()].iloc[-1]
display(best[['ccp_alpha', 'n_leaves', 'roc_auc']])
pruning_rows = []
for label, model in [('Awal', bonus_tree), ('Dipangkas', pruned)]:
    p = model.predict_proba(bonus_X[check])[:, 1]
    pruning_rows.append(dict(model=label, **evaluate(bonus_y[check], p)))
display(pd.DataFrame(pruning_rows))
pruning_path.to_csv(out / 'laporan/tabel/pruning_path.csv', index=False)
print('Pruning hanya memakai calibration/check dari development; tidak membaca holdout final.')


## Blok E2 - Perbandingan pengoptimasi

GD, SGD, dan Adam diuji pada data, penalti, serta batas epoch yang sama. Status converged=False berarti tolerance belum tercapai. Ini eksperimen bonus pada holdout sesudah model final dikunci, bukan pemilihan parameter submission.


In [ ]:
from src.bonus.logreg_optimization import compare_optimizers, plot_optimizer_comparison
opt = compare_optimizers(M[0], yt, M[1], y_val, max_iter=250, batch_size=128,
                         learning_rates={'gd': (.03,.1), 'sgd': (.03,), 'adam': (.03,.1)})
fig = plot_optimizer_comparison(opt)
fig.savefig(out/'laporan/gambar/gambar_6_optimasi.png', dpi=150)
plt.show(); plt.close(fig)
display(opt['results'][['run','epochs','converged','macro_f1','validation_log_loss']])
opt['results'].to_csv(out/'laporan/tabel/optimizer_results.csv', index=False)
opt['history'].to_csv(out/'laporan/tabel/optimizer_history.csv', index=False)


## Blok E3 - Animasi logistik

Model pendamping memakai satu fitur dan intercept tanpa penalti. Kontur dan lintasan berasal dari objective yang sama. Bingkai terakhir disimpan khusus untuk Animasi 1 pada laporan cetak.

In [ ]:
from src.bonus.logreg_video import make_log_loss_video
video = make_log_loss_video(M[0], yt, out / 'laporan/animasi/animasi_logistik.gif',
                            feature_index=0, max_iter=100, max_frames=24, grid_points=25)
mp4 = make_log_loss_video(M[0], yt, out/'laporan/animasi/animasi_logistik.mp4', feature_index=0,
                           max_iter=100, max_frames=24, grid_points=25)
print('GIF:', video['path'], '| video:', mp4['path'])
print('Loss awal/akhir:', video['model'].history_['train_log_loss'][0], video['model'].history_['train_log_loss'][-1])
with Image.open(video['path']) as animation:
    assert animation.n_frames == video['frames']
    animation.seek(animation.n_frames - 1)
    animation.convert('RGB').save(out / 'laporan/animasi/animasi_logistik_akhir.png')
trajectory = video['model'].parameter_history_
pd.DataFrame({'epoch': video['model'].history_['epoch'], 'theta0': trajectory[:, 0],
              'theta1': trajectory[:, 1], 'log_loss': video['model'].history_['train_log_loss']}).to_csv(out / 'laporan/tabel/animation_history.csv', index=False)
print('Frame:', video['frames'], '| FPS:', video['fps'])

display(NotebookImage(filename=str(video['path'])))


## Blok E4 - Bidang keputusan SVM

Eksperimen pendamping memakai dua fitur tertransformasi dan 400 baris training stratified.

In [ ]:
from src.bonus.svm_plot import plot_svm_kernels
viz, _ = train_test_split(np.arange(len(yt)), train_size=min(400, len(yt)-1),
                          stratify=yt, random_state=Settings.SEED)
transformed_names = list(small_matrix.get_feature_names_out())
viz_feature_idx = [transformed_names.index('skewed__' + col) for col in skewed_cols[:2]]
fig = plot_svm_kernels(M[0][viz][:, viz_feature_idx], yt[viz],
                       feature_names=[col + ' (log1p, z-score)' for col in skewed_cols[:2]],
                       path=out / 'laporan/gambar/gambar_7_kernel.png')
plt.show()
plt.close(fig)


## Blok E5 - Graf Bayes

In [ ]:
from src.bonus.naive_bayes_graph import plot_naive_bayes_graph, feature_strengths
bayes_prep, bayes, _ = fitted['optimized', 'Bayes']
fig = plot_naive_bayes_graph(bayes, bayes_prep.feature_names_, top_n=8,
                            path=out/'laporan/gambar/gambar_8_graf_bayes.png')
plt.show(); plt.close(fig)
strengths = feature_strengths(bayes, bayes_prep.feature_names_)
print(sorted(strengths.items(), key=lambda item: -item[1])[:3])


## Blok E6 - Analisis biaya

FP bernilai 2 dan FN bernilai 1. Threshold empiris dipilih pada cal, kemudian diperiksa
pada check yang terpisah. Kebijakan 2/3 merupakan acuan Bayes hanya bila peluang
merupakan posterior yang terkalibrasi; frekuensi pohon berbobot belum tentu memenuhi
asumsi itu. Hasil biaya ini tidak mengubah model atau threshold submission Kaggle.


In [ ]:
from src.bonus.cost_sensitive import cost_sensitive_analysis, plot_cost_sensitive
cost = cost_sensitive_analysis(bonus_tree, bonus_X[cal], bonus_y[cal],
                               bonus_X[check], bonus_y[check])
fig = plot_cost_sensitive(cost)
fig.savefig(out / 'laporan/gambar/gambar_9_biaya.png', dpi=150)
plt.show()
plt.close(fig)
display(cost['results'].query("split == 'test'")[['policy', 'threshold', 'fp', 'fn', 'total_cost']])
cost['results'].to_csv(out / 'laporan/tabel/cost_results.csv', index=False)
cost['curve'].to_csv(out / 'laporan/tabel/cost_threshold_curve.csv', index=False)
expected_figures = [out / filename for filename in (
    'laporan/gambar/gambar_1_kelas.png', 'laporan/gambar/gambar_2_f1.png', 'laporan/gambar/gambar_3_kesalahan.png',
    'laporan/gambar/gambar_4_pohon.png', 'laporan/gambar/gambar_5_pemangkasan.png', 'laporan/gambar/gambar_6_optimasi.png',
    'laporan/gambar/gambar_7_kernel.png', 'laporan/gambar/gambar_8_graf_bayes.png', 'laporan/gambar/gambar_9_biaya.png')]
for filename in expected_figures + [out / 'laporan/animasi/animasi_logistik.gif', out / 'laporan/animasi/animasi_logistik_akhir.png']:
    assert filename.is_file() and filename.stat().st_size > 0
print('Gambar 1-9, GIF, dan bingkai terakhir tersedia.')


---

# Saving and Loading Models <a name="9"></a>

---

Kelima model scratch disimpan dan dibaca ulang. Probabilitas sesudah load harus identik.


In [ ]:
saved_models = {}
for name, own, lib, own_data, lib_data in models:
    model_path = own.save(Settings.OUTPUT_DIR / 'models' / (name.lower().replace('.', '') + '.pkl'))
    loaded = type(own).load(model_path)
    np.testing.assert_array_equal(own.predict_proba(own_data[1]), loaded.predict_proba(own_data[1]))
    saved_models[name] = str(model_path)
print('Save/load kelima model scratch: probabilitas identik.')


---

# Kaggle Submission <a name="10"></a>

---

Model final berasal dari mean CV development, dengan threshold Macro F1 OOF.
Holdout dan threshold biaya FP/FN tidak menentukan pilihan submission. Seluruh baris
train.csv dipakai untuk refit, termasuk booking sah dengan ID berbeda. Preprocessing
hanya di-fit pada train. Artefak menyimpan model, preprocessor, threshold, konfigurasi,
hash input, dan versi dependency. Submission disimpan lokal; tidak ada upload otomatis.


In [ ]:
import sklearn
metadata = {'seed': Settings.SEED, 'metric': 'macro_f1', 'kaggle_metric_verified': False,
            'input_sha256': input_sha256,
            'source_sha256': {str(p.relative_to(project_root)): hashlib.sha256(p.read_bytes()).hexdigest()
                              for p in sorted((project_root/'src').rglob('*.py'))},
            'python': platform.python_version(),
            'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__,
            'selection': json.loads((experiment_dir/'selection.json').read_text(encoding='utf-8')),
            'cv': cv_summary.set_index('key').loc[final_candidate.key].to_dict(),
            'holdout': holdout_scores.query("family == @final_candidate.family and phase == 'optimized' and threshold_kind == 'oof'").to_dict('records')}
submission, final_bundle, submission_metadata = save_final(
    final_candidate, final_threshold, train, test, sample_submission, out, metadata)
assert len(submission) == len(test) == 6844
assert not np.array_equal(submission[TARGET], sample_submission[TARGET])
validate_submission(submission, sample_submission, test)
submission_sha256 = submission_metadata['sha256']
print('Model:', type(final_bundle['model']).__name__, '| threshold:', final_threshold)
print('Refit:', submission_metadata['n_train'], '| submission:', len(submission), '| SHA256:', submission_sha256)
display(submission.head())


---

# Error Analysis <a name="11"></a>

---

Hasil holdout berikut merupakan evaluasi setelah pemilihan model, parameter, dan
threshold dikunci. Jangan memilih ulang model berdasarkan tabel ini. Penurunan hasil
pada holdout dilaporkan apa adanya. CV untuk pemilihan model memakai threshold 0,5;
OOF tuned F1 bukan estimasi holdout. Skor leaderboard belum diketahui.
Kalibrasi SVM dengan subset dan perbedaan distribusi Gaussian/kategori Bayes
membatasi kesetaraan perbandingan dengan scikit-learn.


In [ ]:
display(holdout_scores[['family','phase','threshold_kind','accuracy','macro_f1','roc_auc','pr_auc']].round(4))
selected_cv = cv_summary.set_index('key').loc[final_candidate.key]
print('Pilihan CV:', final_candidate.name, '| mean:', selected_cv.macro_f1_mean,
      '| std:', selected_cv.macro_f1_std, '| OOF threshold:', final_threshold)
p_final_holdout = fitted['optimized', final_candidate.family][2]
errors = val_set[['booking_channel']].copy()
errors['error'] = ((p_final_holdout >= final_threshold).astype(int) != y_val)
display(errors.groupby('booking_channel', dropna=False).error.agg(['count','mean']))
configuration_rows = [dict(key=c.key, balanced=c.balanced, configuration=json.dumps(asdict(c), sort_keys=True))
                       for c in search.candidates.values()]
class_balance = pd.DataFrame(configuration_rows).merge(cv_summary.drop(columns='configuration'), on='key')
class_balance.to_csv(out/'laporan/tabel/class_balance.csv', index=False)



## Referensi dan catatan pengumpulan

Implementasi optimizer tambahan mengacu pada [Kingma & Ba, Adam (2014/ICLR 2015)](https://arxiv.org/abs/1412.6980) dan [Parikh & Boyd, Proximal Algorithms](https://web.stanford.edu/~boyd/papers/pdf/prox_algs.pdf).
Notebook ini dilengkapi dengan bantuan Codex; bantuan implementasi/eksperimen perlu dicatat secara jujur pada Form Penggunaan Generative AI kelompok. Grafik dan tabel berasal dari run aktual, tanpa klaim skor leaderboard. Unggahan Kaggle dan penyisipan visual ke PDF laporan dilakukan oleh kelompok.
Partisi kategori biner CART mengacu pada [Hastie, Tibshirani & Friedman, ESL ?9.2.4](https://tibshirani.su.domains/ElemStatLearn.1stEd/errata/errata1.pdf).
